# wrap-forward-fn-generic — ex3: wrap_forward_fn passes non-array outputs through un-boxed

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `wrap-forward-fn-generic`. Running the final beacon cell reports progress against the `Backprop: wrap forward fn` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: wrap forward fn` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wrap-forward-fn-generic`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wrap-forward-fn-generic"
DD_SUBTOPIC = "Backprop: wrap forward fn"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## wrap_forward_fn — quick refresher

`wrap_forward_fn(fwd_fn)` returns a Tensor-aware wrapper. The happy path is **unbox → call → box**: pull `.array` out of Tensor inputs, invoke the raw `fwd_fn`, wrap the array result back into a `Tensor`.

**This drill (ex3) vs prior.** ex1 implemented the shell — assuming `fwd_fn` always returns a tensor. ex2 added kwargs pass-through and the `is_differentiable` short-circuit. NEITHER handled the case where `fwd_fn` returns a *non-array* — a Python int, a tuple, a bool. Boxing those into a `Tensor` either crashes (`Tensor(int)` does the wrong thing) or hides a real bug. The ARENA wrapper must pass non-array outputs through UN-BOXED.

### Exercise 3 — wrap_forward_fn passes non-array outputs through un-boxed

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Create a `wrap_forward_fn` variant that boxes array-like results as Tensor but passes non-array results (Python scalars, ints, tuples) through unchanged.
> Keywords: non-tensor-output, argmax, scalar-return, pass-through
> ```

**KCs targeted:** `wrap-forward-fn-generic`, `non-tensor-output-pass-through`

Implement `wrap_forward_fn(fwd_fn)`. The wrapper closure must:

1. **Unbox** — for each positional arg, if it is a `Tensor` instance (the minimal class given below), replace with `.array`. Pass everything else through unchanged.
2. **Call** — `result = fwd_fn(*raw_args, **kwargs)`.
3. **Conditional box** — if `result` is a `torch.Tensor` (i.e. `isinstance(result, t.Tensor)`), return `Tensor(result)`. OTHERWISE return `result` unchanged (no boxing).

**Why this matters.** Some ARENA-eligible forward fns don't return arrays:
- `lambda x: int(x.argmax().item())` → returns `int`.
- `lambda x: x.shape` → returns `torch.Size` (a tuple subclass, not a Tensor).
- `lambda x: x.numel()` → returns `int`.

Forcing every output through `Tensor(...)` either errors (`Tensor(some_int)` doesn't have an `.array`) or produces a 0-D Tensor that breaks downstream control flow expecting a real Python int.

The minimal `Tensor` class is given:
```python
class Tensor:
    def __init__(self, array):
        self.array = array
```
(Definitions are pasted into the solution; the test relies on your wrap_forward_fn working with this exact class.)

In [ ]:
class Tensor:
    def __init__(self, array):
        self.array = array


def wrap_forward_fn(fwd_fn):
    def tensor_func(*args, **kwargs):
        raw_args = [a.array if isinstance(a, Tensor) else a for a in args]
        result = fwd_fn(*raw_args, **kwargs)
        if isinstance(result, t.Tensor):
            return Tensor(result)
        return result
    return tensor_func


<details><summary>Solution</summary>

```python
class Tensor:
    def __init__(self, array):
        self.array = array


def wrap_forward_fn(fwd_fn):
    def tensor_func(*args, **kwargs):
        raw_args = [a.array if isinstance(a, Tensor) else a for a in args]
        result = fwd_fn(*raw_args, **kwargs)
        if isinstance(result, t.Tensor):
            return Tensor(result)
        return result
    return tensor_func
```

**Why an `isinstance(result, t.Tensor)` check, not `hasattr(result, 'shape')`.** A `torch.Size` has `.shape`-like behavior but is a tuple, and tuples have no `.array` semantics. Type-checking on `t.Tensor` is the precise discriminator.

**Why NOT box ints into 0-D Tensors.** A Python-int output from `argmax` or `numel` is usually consumed by code like `for i in range(n):` or `xs[idx]`. A 0-D Tensor doesn't satisfy those patterns without an extra `.item()` call. Pass-through preserves the original semantics.

**The is_differentiable extension from ex2 still applies.** When you combine ex2 and ex3, the order of operations is: (a) unbox; (b) call fwd_fn; (c) IF result is a Tensor AND is_differentiable AND any input had requires_grad → attach a Recipe; (d) ELSE pass result through. Non-tensor outputs are non-differentiable by construction.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()